# FreshGuard v2 - 01 Dataset Audit And Split

Inputs required on Kaggle:
- `ulnnproject/food-freshness-dataset`
- notebook 00 output saved as `freshguard-official-sources-v2`

Food Freshness is the canonical 24-class freshness benchmark. KTH GroceryStoreDataset is parsed as official external type/generalization evidence and optional `fresh_assumed` auxiliary training rows; it is not mixed into headline freshness test metrics.

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import pandas as pd
from PIL import Image, UnidentifiedImageError

try:
    import imagehash
except ImportError as exc:
    raise RuntimeError('Install imagehash in the Kaggle session before running notebook 01.') from exc

SEED = 42
random.seed(SEED)
KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/freshguard_v2_splits')
WORK_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
PRODUCE_TYPES = ('apple', 'banana', 'bellpepper', 'bitter_gourd', 'carrot', 'cucumber', 'mango', 'okra', 'orange', 'potato', 'strawberry', 'tomato')
FRESHNESS_LEVELS = ('fresh', 'rotten')
COMBINED_CLASSES = tuple(f'{produce}_{freshness}' for produce in PRODUCE_TYPES for freshness in FRESHNESS_LEVELS)
TYPE_ALIASES = {
    'apple': 'apple', 'apples': 'apple', 'royalgala': 'apple', 'grannysmith': 'apple', 'goldendelicious': 'apple',
    'banana': 'banana', 'bananas': 'banana', 'orange': 'orange', 'oranges': 'orange',
    'tomato': 'tomato', 'tomatoes': 'tomato', 'potato': 'potato', 'potatoes': 'potato',
    'carrot': 'carrot', 'carrots': 'carrot', 'cucumber': 'cucumber', 'cucumbers': 'cucumber',
    'mango': 'mango', 'mangoes': 'mango', 'okra': 'okra', 'okara': 'okra',
    'strawberry': 'strawberry', 'strawberries': 'strawberry',
    'bellpepper': 'bellpepper', 'bellpeppers': 'bellpepper', 'bellpepperred': 'bellpepper', 'bellpeppergreen': 'bellpepper', 'bellpepperyellow': 'bellpepper', 'bell_pepper': 'bellpepper', 'capsicum': 'bellpepper', 'capciscum': 'bellpepper', 'pepper': 'bellpepper', 'redbellpepper': 'bellpepper', 'greenbellpepper': 'bellpepper',
    'bittergourd': 'bitter_gourd', 'bittergroud': 'bitter_gourd', 'bitter_gourd': 'bitter_gourd',
}
FRESHNESS_ALIASES = {'fresh': 'fresh', 'purefresh': 'fresh', 'good': 'fresh', 'healthy': 'fresh', 'ripe': 'fresh', 'rotten': 'rotten', 'rotton': 'rotten', 'bad': 'rotten', 'spoiled': 'rotten', 'stale': 'rotten'}

def tokens(value: str) -> list[str]:
    import re
    return [token for token in re.split(r'[^a-z0-9]+', value.lower()) if token]

def collapse(value: str) -> str:
    return ''.join(tokens(value))

def normalize_type(value: str) -> str | None:
    collapsed = collapse(value)
    if collapsed in TYPE_ALIASES:
        return TYPE_ALIASES[collapsed]
    for alias, produce in sorted(TYPE_ALIASES.items(), key=lambda item: len(item[0]), reverse=True):
        if alias in collapsed:
            return produce
    return None

def normalize_freshness(value: str) -> str | None:
    collapsed = collapse(value)
    if collapsed in FRESHNESS_ALIASES:
        return FRESHNESS_ALIASES[collapsed]
    for token in tokens(value):
        if token in FRESHNESS_ALIASES:
            return FRESHNESS_ALIASES[token]
    return None

def find_first_existing(candidates: list[Path], must_contain: str) -> Path:
    for root in candidates:
        if root.exists() and any(root.rglob(must_contain)):
            return root
    raise FileNotFoundError(f'Could not find input containing {must_contain!r}: {[str(c) for c in candidates]}')

def find_first_image_root(candidates: list[Path]) -> Path:
    for root in candidates:
        if root.exists() and any(p.suffix.lower() in IMAGE_EXTENSIONS for p in root.rglob('*')):
            return root
    raise FileNotFoundError(f'Could not find image dataset root in: {[str(c) for c in candidates]}')

def find_official_source_root() -> Path:
    summaries = sorted(KAGGLE_INPUT.rglob('official_sources_summary.json'))
    if not summaries:
        raise FileNotFoundError(
            'Could not find official_sources_summary.json anywhere under /kaggle/input. '
            'Attach the output dataset from kaggle_00_fetch_official_sources_v2.ipynb.'
        )
    return summaries[0].parent

food_candidates = [p for p in KAGGLE_INPUT.glob('*') if p.is_dir() and 'official' not in p.name.lower() and 'freshguard' not in p.name.lower()]
preferred_food = [p for p in food_candidates if 'fresh' in p.name.lower() or 'food' in p.name.lower()]
FOOD_ROOT = find_first_image_root(preferred_food + food_candidates)
OFFICIAL_ROOT = find_official_source_root()
GROCERY_ROOT = next(OFFICIAL_ROOT.rglob('grocery_store_dataset'))
print({'food_root': str(FOOD_ROOT), 'official_root': str(OFFICIAL_ROOT), 'grocery_root': str(GROCERY_ROOT)})

In [ ]:
@dataclass(frozen=True)
class ManifestRow:
    path: str
    source: str
    split: str
    produce_type: str
    freshness: str
    combined_label: str
    supervision: str
    phash: str
    width: int
    height: int
    cluster_id: int | None = None

def iter_images(root: Path) -> list[Path]:
    return [p for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS]

def infer_food_labels(path: Path) -> tuple[str | None, str | None]:
    candidates = [path.stem, *reversed(path.parts[:-1])]
    produce = next((p for value in candidates if (p := normalize_type(value))), None)
    freshness = next((f for value in candidates if (f := normalize_freshness(value))), None)
    return produce, freshness

def image_meta(path: Path) -> tuple[str, int, int]:
    with Image.open(path) as img:
        rgb = img.convert('RGB')
        phash = str(imagehash.phash(rgb))
        width, height = rgb.size
    return phash, width, height

food_rows: list[dict[str, object]] = []
rejects: list[dict[str, str]] = []
for path in iter_images(FOOD_ROOT):
    produce, freshness = infer_food_labels(path)
    if produce is None or freshness is None:
        rejects.append({'path': str(path), 'source': 'food_freshness', 'reason': 'label_inference_failed'})
        continue
    combined = f'{produce}_{freshness}'
    if combined not in COMBINED_CLASSES:
        rejects.append({'path': str(path), 'source': 'food_freshness', 'reason': 'unsupported_label'})
        continue
    try:
        phash, width, height = image_meta(path)
    except (OSError, UnidentifiedImageError) as exc:
        rejects.append({'path': str(path), 'source': 'food_freshness', 'reason': f'decode_failed:{exc.__class__.__name__}'})
        continue
    food_rows.append({'path': str(path), 'source': 'food_freshness', 'produce_type': produce, 'freshness': freshness, 'combined_label': combined, 'supervision': 'freshness_labeled', 'phash': phash, 'width': width, 'height': height})

food_df = pd.DataFrame(food_rows)
if food_df.empty:
    raise RuntimeError('Food Freshness audit accepted zero rows. Check Kaggle input attachment.')
print({'food_rows': len(food_df), 'rejects': len(rejects)})

In [ ]:
def hamming_hex(a: str, b: str) -> int:
    return bin(int(a, 16) ^ int(b, 16)).count('1')

parent = list(range(len(food_df)))
def find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a: int, b: int) -> None:
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for _, group in food_df.groupby('combined_label'):
    indices = list(group.index)
    hashes = food_df.loc[indices, 'phash'].tolist()
    for i, idx_i in enumerate(indices):
        for idx_j, hash_j in zip(indices[i + 1:], hashes[i + 1:], strict=False):
            if hamming_hex(hashes[i], hash_j) <= 5:
                union(int(idx_i), int(idx_j))

food_df['cluster_id'] = [find(i) for i in range(len(food_df))]
clustered = food_df.drop_duplicates('cluster_id').copy()
split_by_cluster: dict[int, str] = {}
for _, group in clustered.groupby('combined_label'):
    cluster_ids = list(group['cluster_id'])
    random.shuffle(cluster_ids)
    n = len(cluster_ids)
    n_train = int(round(n * 0.70))
    n_val = int(round(n * 0.15))
    for cid in cluster_ids[:n_train]:
        split_by_cluster[int(cid)] = 'train'
    for cid in cluster_ids[n_train:n_train + n_val]:
        split_by_cluster[int(cid)] = 'val'
    for cid in cluster_ids[n_train + n_val:]:
        split_by_cluster[int(cid)] = 'test'
food_df['split'] = food_df['cluster_id'].map(lambda cid: split_by_cluster[int(cid)])
food_df.to_csv(WORK_DIR / 'classifier_manifest_food_freshness_only.csv', index=False)

In [ ]:
classes_csv = GROCERY_ROOT / 'dataset' / 'classes.csv'
if not classes_csv.exists():
    raise FileNotFoundError(f'Missing KTH classes.csv: {classes_csv}')
classes_df = pd.read_csv(classes_csv)
required_class_cols = {'Class Name (str)', 'Class ID (int)', 'Coarse Class Name (str)'}
if required_class_cols.issubset(classes_df.columns):
    class_names = [str(value) for value in classes_df['Class Name (str)'].tolist()]
    index_to_type = {}
    for _, class_row in classes_df.iterrows():
        fine_name = str(class_row['Class Name (str)'])
        coarse_name = str(class_row['Coarse Class Name (str)'])
        produce = normalize_type(fine_name) or normalize_type(coarse_name)
        if produce is not None:
            index_to_type[int(class_row['Class ID (int)'])] = produce
else:
    classes_df = pd.read_csv(classes_csv, header=None)
    class_names = [str(value) for value in classes_df.iloc[:, 0].tolist()]
    index_to_type = {index: produce for index, raw_name in enumerate(class_names) if (produce := normalize_type(raw_name)) is not None}

def parse_kth_split(split_name: str) -> list[dict[str, object]]:
    split_file = GROCERY_ROOT / 'dataset' / f'{split_name}.txt'
    rows: list[dict[str, object]] = []
    if not split_file.exists():
        return rows
    for line in split_file.read_text().splitlines():
        parts = [part.strip() for part in line.strip().split(',')]
        if len(parts) < 3:
            continue
        rel_path = parts[0]
        try:
            fine_label = int(parts[1])
        except ValueError:
            fine_label = class_names.index(parts[1]) if parts[1] in class_names else -1
        produce = index_to_type.get(fine_label)
        if produce is None:
            continue
        path = GROCERY_ROOT / 'dataset' / rel_path
        if not path.exists():
            path = GROCERY_ROOT / rel_path
        if not path.exists():
            continue
        try:
            phash, width, height = image_meta(path)
        except (OSError, UnidentifiedImageError):
            continue
        rows.append({'path': str(path), 'source': 'grocery_store_dataset', 'split': split_name, 'produce_type': produce, 'freshness': 'fresh', 'combined_label': f'{produce}_fresh', 'supervision': 'fresh_assumed', 'phash': phash, 'width': width, 'height': height, 'cluster_id': None})
    return rows

grocery_rows = parse_kth_split('train') + parse_kth_split('val') + parse_kth_split('test')
grocery_df = pd.DataFrame(grocery_rows)
if grocery_df.empty:
    raise RuntimeError('KTH GroceryStoreDataset produced zero mapped produce rows.')
grocery_external = grocery_df[grocery_df['split'] == 'test'].copy()
grocery_aux = grocery_df[grocery_df['split'].isin(['train', 'val'])].copy()
classifier_manifest = pd.concat([food_df, grocery_aux], ignore_index=True, sort=False)
classifier_manifest.to_csv(WORK_DIR / 'classifier_manifest.csv', index=False)
grocery_external.to_csv(WORK_DIR / 'grocery_external_eval_manifest.csv', index=False)
pd.concat([food_df, grocery_df], ignore_index=True, sort=False).to_csv(WORK_DIR / 'source_manifest_raw.csv', index=False)
pd.DataFrame(rejects).to_csv(WORK_DIR / 'rejects.csv', index=False)
print({'classifier_rows': len(classifier_manifest), 'kth_aux_rows': len(grocery_aux), 'kth_external_test_rows': len(grocery_external)})

In [ ]:
summary = {
    'food_rows': int(len(food_df)),
    'food_clusters': int(food_df['cluster_id'].nunique()),
    'food_by_split': food_df['split'].value_counts().to_dict(),
    'kth_aux_rows': int(len(grocery_aux)),
    'kth_external_test_rows': int(len(grocery_external)),
    'classifier_manifest': 'classifier_manifest.csv',
    'canonical_food_manifest': 'classifier_manifest_food_freshness_only.csv',
    'grocery_external_eval_manifest': 'grocery_external_eval_manifest.csv',
}
(WORK_DIR / 'split_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)
print('Save /kaggle/working/freshguard_v2_splits as Kaggle Dataset: freshguard-v2-splits')